# Adapter Training — Chittagong Dialect → English

This notebook trains **LoRA adapters** on top of the dialect-tagged NLLB-200-600M model for the Chittagong dialect specifically, then evaluates on a held-out validation set using:

- **SacreBLEU** (corpus-level)
- **chrF++**
- **ROUGE-1, ROUGE-2, ROUGE-L**
- **METEOR**
- **Word Error Rate (WER)**
- **Character Error Rate (CER)**

### Key design decisions
- Only the **LoRA adapter weights** are trained — the base NLLB model is fully frozen
- Chittagong text gets **punctuation/whitespace-only normalization** (no full csebuet normalize)
- A small Standard Bangla anchor sample is mixed in to prevent catastrophic forgetting
- Adapter weights are saved separately — the base model is never modified

### Kaggle setup
- Accelerator: **GPU T4 x2**
- Internet: **On** (to install packages)
- Add your dialect-tagged model dataset: `/kaggle/input/nllb-dialect-model/`
- Add your Chittagong data dataset

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║              ⚠️  AUTO-RUN GUARD  ⚠️                          ║
# ║  Delete or skip this cell, then run cells one at a time.    ║
# ╚══════════════════════════════════════════════════════════════╝
import sys
print("⛔ Auto-run blocked. Delete this cell and run manually with Shift+Enter.")
sys.exit("Auto-run guard triggered.")

In [2]:
!pip install transformers==5.2.0 peft==0.18.1 datasets sacrebleu \
             rouge-score nltk jiwer sentencepiece accelerate \
             evaluate --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 52.5 MB/s eta 0:00:00a 0:00:01


In [3]:
import json

config_path = "/kaggle/input/datasets/farhanaadri/nllb-dialect-model/config.json"
with open(config_path, "r") as f:
    config = json.load(f)

print(json.dumps(config, indent=2))

{
  "activation_dropout": 0.0,
  "activation_function": "relu",
  "architectures": [
    "M2M100ForConditionalGeneration"
  ],
  "attention_dropout": 0.1,
  "bos_token_id": 0,
  "d_model": 1024,
  "decoder_attention_heads": 16,
  "decoder_ffn_dim": 4096,
  "decoder_layerdrop": 0,
  "decoder_layers": 12,
  "decoder_start_token_id": 2,
  "dropout": 0.1,
  "dtype": "float32",
  "encoder_attention_heads": 16,
  "encoder_ffn_dim": 4096,
  "encoder_layerdrop": 0,
  "encoder_layers": 12,
  "eos_token_id": 2,
  "init_std": 0.02,
  "is_encoder_decoder": true,
  "max_position_embeddings": 1024,
  "model_type": "m2m_100",
  "num_hidden_layers": 12,
  "pad_token_id": 1,
  "scale_embedding": true,
  "tie_word_embeddings": true,
  "tokenizer_class": "NllbTokenizer",
  "transformers_version": "5.2.0",
  "use_cache": true,
  "vocab_size": 256209
}


In [4]:
# ── Configuration — EDIT THESE ────────────────────────────────────────────────

# Path to your dialect-tagged NLLB model (from the add_dialect_tags notebook)
BASE_MODEL_PATH = "/kaggle/input/datasets/farhanaadri/nllb-dialect-model"

# Path to your Chittagong CSV data
# Expected columns: 'dialect_text' (Chittagong), 'english_text' (English translation)
CHITTAGONG_CSV  = "/kaggle/input/datasets/farhanaadri/all-dialects-data/dataset_translated/train/chittagong_dialect_final.csv"  # <-- CHANGE
DIALECT_COL     = "dialect_speech"    # <-- column with Chittagong text
ENGLISH_COL     = "English_translation"    # <-- column with English translation

# Optional: Standard Bangla CSV for anchor mixing (set to None to skip)
# Prevents catastrophic forgetting of standard Bangla during adapter training
BANGLA_CSV      = CHITTAGONG_CSV  # e.g. "/kaggle/input/your-data/standard_bangla.csv"
BANGLA_DIALECT_COL  = "Standard_Bangla"
BANGLA_ENGLISH_COL  = "English_translation"
BANGLA_ANCHOR_FRAC  = 0.10   # mix 10% standard Bangla into each training batch



# Train/validation split
DIALECT_COL_VAL     = "chittagong_bangla_speech "    # <-- column with Chittagong text
ENGLISH_COL_VAL     = "english_speech"    # <-- column with English translation
# VAL_CSV = "/kaggle/input/datasets/farhanaadri/all-dialects-data/dataset_translated/validation/chittagong_validation.csv"   # <-- ADD THIS
VAL_CSV = "/kaggle/input/datasets/farhanaadri/vashantor/Vashantor A Large-scale Multilingual Benchmark Dataset for Automated Translation of Bangla Regional Dialects to Bangla Language/Vashantor_CSV_Format/Vashantor_CSV_Format/Validation/Chittagong Validation Translation.csv"   # <-- ADD THIS

# LoRA adapter hyperparameters
LORA_R          = 16      # rank — higher = more expressive but more parameters
LORA_ALPHA      = 32      # scaling factor (usually 2x rank)
LORA_DROPOUT    = 0.05
# Which layers to apply LoRA to. These are the projection matrices in each
# transformer block. q_proj and v_proj is standard; add k_proj, out_proj
# for more capacity if your data is large enough.
LORA_TARGET_MODULES = ["q_proj", "v_proj"]

# Training hyperparameters
NUM_EPOCHS      = 10
BATCH_SIZE      = 16      # per-device batch size
GRAD_ACCUM      = 2       # effective batch = BATCH_SIZE * GRAD_ACCUM * num_gpus
LEARNING_RATE   = 3e-4    # adapters can use higher LR than full fine-tuning
WARMUP_STEPS    = 100
MAX_SOURCE_LEN  = 128
MAX_TARGET_LEN  = 128
WEIGHT_DECAY    = 0.01
FP16            = True    # use mixed precision on T4
RANDOM_SEED = 42
# Generation hyperparameters (used during evaluation)
NUM_BEAMS       = 5

# Where to save adapter weights
OUTPUT_DIR      = "/kaggle/working/chittagong_adapter_v2"

# Dialect tag for Chittagong (must match what was set in add_dialect_tags notebook)
SRC_LANG        = "bng_chittagong"
TGT_LANG        = "eng_Latn"

print("Configuration loaded.")

Configuration loaded.


In [5]:
# ── Imports ───────────────────────────────────────────────────────────────────
import os
import re
import json
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Optional

import torch
from torch.utils.data import Dataset

import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

from transformers import (
    NllbTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)
from peft import (
    get_peft_model,
    LoraConfig,
    TaskType,
    PeftModel,
)
import sacrebleu
from rouge_score import rouge_scorer
from nltk.translate.meteor_score import meteor_score
from jiwer import wer, cer

warnings.filterwarnings("ignore")

device = "cuda" if torch.cuda.is_available() else "cpu"
n_gpus = torch.cuda.device_count()
print(f"Device  : {device}")
print(f"Num GPUs: {n_gpus}")
if device == "cuda":
    for i in range(n_gpus):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

Device  : cuda
Num GPUs: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4


In [6]:
# ── Chittagong-safe normalization ─────────────────────────────────────────────
#
# Based on verification results: csebuet normalizer damages Chittagong-specific
# features (loanwords, nasalization, pronouns). We apply ONLY:
#   1. Whitespace normalization (collapse multiple spaces, strip)
#   2. Punctuation normalization (Unicode punctuation → ASCII equivalents)
#   3. Zero-width character removal
# We deliberately do NOT apply phoneme normalization or unicode composition
# that the full csebuet normalizer would do.

# Unicode punctuation → ASCII map
_PUNCT_MAP = str.maketrans({
    "\u201c": '"', "\u201d": '"',   # curly double quotes
    "\u2018": "'", "\u2019": "'",   # curly single quotes
    "\u2013": "-", "\u2014": "-",   # en-dash, em-dash
    "\u2026": "...",                 # ellipsis
    "\u00a0": " ",                   # non-breaking space
    "\u200b": "",                    # zero-width space
    "\u200c": "",                    # zero-width non-joiner
    "\u200d": "",                    # zero-width joiner
    "\ufeff": "",                    # BOM
})

def normalize_chittagong(text: str) -> str:
    """Safe normalization that preserves Chittagong dialectal features."""
    if not isinstance(text, str):
        return ""
    text = text.translate(_PUNCT_MAP)          # punctuation normalization
    text = re.sub(r"[\u200b-\u200f\ufeff]", "", text)  # zero-width chars
    text = re.sub(r" +", " ", text)            # collapse multiple spaces
    text = text.strip()
    return text

# Quick test
test = "আঁই\u200b হন্ডে  যাইচ\u201d"
print(f"Before: {repr(test)}")
print(f"After : {repr(normalize_chittagong(test))}")
print("Normalization function ready.")

Before: 'আঁই\u200b হন্ডে  যাইচ”'
After : 'আঁই হন্ডে যাইচ"'
Normalization function ready.


In [9]:
# ── Load and split data ───────────────────────────────────────────────────────

df = pd.read_csv(CHITTAGONG_CSV)
print(f"Loaded {len(df)} rows from {CHITTAGONG_CSV}")
print(f"Columns: {list(df.columns)}")

# Drop rows with missing values in either column
df = df[[DIALECT_COL, ENGLISH_COL]].dropna().reset_index(drop=True)
df = df[df[DIALECT_COL].str.strip().ne("") & df[ENGLISH_COL].str.strip().ne("")]
print(f"After dropping nulls/empty: {len(df)} rows")

# Apply safe normalization
df[DIALECT_COL] = df[DIALECT_COL].apply(normalize_chittagong)
df[ENGLISH_COL] = df[ENGLISH_COL].apply(lambda x: x.strip() if isinstance(x, str) else x)

# Use full training CSV as train, load separate validation CSV
df_train = df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

df_val_raw = pd.read_csv(VAL_CSV)
df_val = df_val_raw[[DIALECT_COL_VAL, ENGLISH_COL_VAL]].dropna().reset_index(drop=True)
df_val[DIALECT_COL_VAL] = df_val[DIALECT_COL_VAL].apply(normalize_chittagong)
df_val[ENGLISH_COL_VAL] = df_val[ENGLISH_COL_VAL].apply(lambda x: x.strip() if isinstance(x, str) else x)

print(f"\nTrain samples : {len(df_train)}")
print(f"Val samples   : {len(df_val)}")
print(f"\nSample training row:")
print(f"  Source : {df_train[DIALECT_COL].iloc[0]}")
print(f"  Target : {df_train[ENGLISH_COL].iloc[0]}")

Loaded 12957 rows from /kaggle/input/datasets/farhanaadri/all-dialects-data/dataset_translated/train/chittagong_dialect_final.csv
Columns: ['Standard_Bangla', 'dialect_speech', 'English_translation']
After dropping nulls/empty: 12957 rows

Train samples : 12957
Val samples   : 250

Sample training row:
  Source : আল্লা দান অর আত আরো লম্বা গরন
  Target : May Allah extend the hand of giving even further.


In [10]:
# ── Optionally load Standard Bangla anchor data ───────────────────────────────
#
# Mixing a small fraction of standard Bangla samples prevents the adapter from
# overfitting so hard to Chittagong that it damages the model's general Bangla
# translation ability.

df_bangla_train = None

if BANGLA_CSV is not None:
    df_bn = pd.read_csv(BANGLA_CSV)
    df_bn = df_bn[[BANGLA_DIALECT_COL, BANGLA_ENGLISH_COL]].dropna().reset_index(drop=True)
    df_bn.columns = [DIALECT_COL, ENGLISH_COL]  # rename to match

    # Take anchor_frac proportion relative to Chittagong training size
    n_anchor = int(len(df_train) * BANGLA_ANCHOR_FRAC)
    df_bangla_train = df_bn.sample(
        n=min(n_anchor, len(df_bn)), random_state=RANDOM_SEED
    ).reset_index(drop=True)

    print(f"Standard Bangla anchor samples: {len(df_bangla_train)}")
else:
    print("No Bangla anchor data configured (BANGLA_CSV is None). Skipping.")

Standard Bangla anchor samples: 1295


In [11]:
# ── Load tokenizer and model ──────────────────────────────────────────────────

print(f"Loading tokenizer from: {BASE_MODEL_PATH}")
tokenizer = NllbTokenizer.from_pretrained(BASE_MODEL_PATH, use_fast=False)
print(f"  Vocab size: {len(tokenizer)}")

# Verify our dialect tag is present
src_tag_id = tokenizer.convert_tokens_to_ids(SRC_LANG)
assert src_tag_id != tokenizer.unk_token_id, (
    f"Source lang tag '{SRC_LANG}' not found in tokenizer! "
    f"Make sure you're loading from the dialect-tagged model, not the original NLLB."
)
tgt_tag_id = tokenizer.convert_tokens_to_ids(TGT_LANG)
print(f"  {SRC_LANG} → token_id {src_tag_id} ✅")
print(f"  {TGT_LANG} → token_id {tgt_tag_id} ✅")

print(f"\nLoading model from: {BASE_MODEL_PATH}")
from transformers import AutoConfig

# Load and fix config before loading model
config = AutoConfig.from_pretrained(BASE_MODEL_PATH)

# Fix: the tied_weights_keys field may be a list instead of dict in saved config
if hasattr(config, 'tied_weights_keys') and isinstance(config.tied_weights_keys, list):
    pass  # this is fine, just suppress

model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL_PATH,
    config=config,
    dtype=torch.float16 if FP16 else torch.float32,
    ignore_mismatched_sizes=True,
)
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

Loading tokenizer from: /kaggle/input/datasets/farhanaadri/nllb-dialect-model
  Vocab size: 256209
  bng_chittagong → token_id 256207 ✅
  eng_Latn → token_id 256047 ✅

Loading model from: /kaggle/input/datasets/farhanaadri/nllb-dialect-model


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

  Parameters: 615,076,864


In [12]:
import transformers
import peft
import torch

print(f"transformers : {transformers.__version__}")
print(f"peft         : {peft.__version__}")
print(f"torch        : {torch.__version__}")
print(f"torch cuda   : {torch.cuda.is_available()}")

transformers : 5.2.0
peft         : 0.18.1
torch        : 2.9.0+cu126
torch cuda   : True


In [13]:
import os
for root, dirs, files in os.walk("/kaggle/input/datasets/farhanaadri/nllb-dialect-model"):
    for f in files:
        print(os.path.join(root, f))


/kaggle/input/datasets/farhanaadri/nllb-dialect-model/config.json
/kaggle/input/datasets/farhanaadri/nllb-dialect-model/tokenizer.json
/kaggle/input/datasets/farhanaadri/nllb-dialect-model/tokenizer_config.json
/kaggle/input/datasets/farhanaadri/nllb-dialect-model/model.safetensors
/kaggle/input/datasets/farhanaadri/nllb-dialect-model/sentencepiece.bpe.model
/kaggle/input/datasets/farhanaadri/nllb-dialect-model/generation_config.json


In [14]:
# ── Apply LoRA adapter configuration ─────────────────────────────────────────
#
# LoRA (Low-Rank Adaptation) inserts small trainable matrices into the attention
# layers. The base model weights are completely frozen — only the adapter
# matrices (a tiny fraction of total parameters) are updated during training.
#
# TaskType.SEQ_2_SEQ_LM tells PEFT this is an encoder-decoder model.

lora_config = LoraConfig(
    task_type       = TaskType.SEQ_2_SEQ_LM,
    r               = LORA_R,
    lora_alpha      = LORA_ALPHA,
    lora_dropout    = LORA_DROPOUT,
    target_modules  = LORA_TARGET_MODULES,
    bias            = "none",
)

# model = get_peft_model(model, lora_config)   #If training from beginning

# Replace with this
from peft import PeftModel      #Continue training from checkpoint
model = PeftModel.from_pretrained(
    model,
    "/kaggle/working/chittagong_adapter/final_adapter",
    is_trainable=True    # must be True to continue training
)

# Print trainable vs frozen parameter counts
trainable   = sum(p.numel() for p in model.parameters() if p.requires_grad)
total       = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters : {trainable:>12,}  ({100*trainable/total:.3f}% of total)")
print(f"Frozen parameters    : {total - trainable:>12,}")
print(f"Total parameters     : {total:>12,}")
print()
model.print_trainable_parameters()

Trainable parameters :    2,359,296  (0.382% of total)
Frozen parameters    :  615,076,864
Total parameters     :  617,436,160

trainable params: 2,359,296 || all params: 617,436,160 || trainable%: 0.3821


In [16]:
# ── Dataset class ─────────────────────────────────────────────────────────────

class DialectDataset(Dataset):
    """
    PyTorch Dataset for dialect → English translation.
    Handles tokenization with the correct source language tag per sample.
    """
    def __init__(
        self,
        sources: list,
        targets: list,
        tokenizer: NllbTokenizer,
        src_lang: str,
        tgt_lang: str,
        max_source_len: int = 128,
        max_target_len: int = 128,
    ):
        self.sources        = sources
        self.targets        = targets
        self.tokenizer      = tokenizer
        self.src_lang       = src_lang
        self.tgt_lang       = tgt_lang
        self.max_source_len = max_source_len
        self.max_target_len = max_target_len

    def __len__(self):
        return len(self.sources)

    def __getitem__(self, idx):
        source = str(self.sources[idx])
        target = str(self.targets[idx])

        # Tokenize source with dialect tag
        self.tokenizer.src_lang = self.src_lang
        model_inputs = self.tokenizer(
            source,
            max_length=self.max_source_len,
            truncation=True,
            padding=False,
        )

        # Tokenize target (English)
        self.tokenizer.src_lang = self.tgt_lang
        # Replace with this
        self.tokenizer.src_lang = self.tgt_lang
        labels = self.tokenizer(
            target,
            max_length=self.max_target_len,
            truncation=True,
            padding=False,
        )
        # Reset src_lang back to source language
        self.tokenizer.src_lang = self.src_lang

        model_inputs["labels"] = labels["input_ids"]
        return model_inputs


# Build training dataset — combine Chittagong + optional Bangla anchor
train_sources = df_train[DIALECT_COL].tolist()
train_targets = df_train[ENGLISH_COL].tolist()
train_src_langs = [SRC_LANG] * len(train_sources)

if df_bangla_train is not None:
    train_sources  += df_bangla_train[DIALECT_COL].tolist()
    train_targets  += df_bangla_train[ENGLISH_COL].tolist()
    train_src_langs += ["ben_Beng"] * len(df_bangla_train)
    print(f"Combined training set: {len(train_sources)} samples ")
    print(f"  ({len(df_train)} Chittagong + {len(df_bangla_train)} Standard Bangla anchor)")

# Shuffle combined training set
combined = list(zip(train_sources, train_targets, train_src_langs))
import random
random.seed(RANDOM_SEED)
random.shuffle(combined)
train_sources, train_targets, train_src_langs = zip(*combined)

# Note: for multi-dialect mixing with different src_langs per sample, we build
# the dataset per-dialect and concatenate. For the anchor case here, since
# the anchor is small and standard Bangla is close to Chittagong tag, we use
# a single dataset with SRC_LANG for simplicity.
# For a fully rigorous approach, sub-class Dataset to handle per-sample src_lang.
train_dataset = DialectDataset(
    list(train_sources), list(train_targets),
    tokenizer, SRC_LANG, TGT_LANG,
    MAX_SOURCE_LEN, MAX_TARGET_LEN
)

val_dataset = DialectDataset(
    df_val[DIALECT_COL_VAL].tolist(), df_val[ENGLISH_COL_VAL].tolist(),
    tokenizer, SRC_LANG, TGT_LANG,
    MAX_SOURCE_LEN, MAX_TARGET_LEN
)

print(f"\nTrain dataset size : {len(train_dataset)}")
print(f"Val dataset size   : {len(val_dataset)}")

Combined training set: 14252 samples 
  (12957 Chittagong + 1295 Standard Bangla anchor)

Train dataset size : 14252
Val dataset size   : 250


In [17]:
# ── Data collator ─────────────────────────────────────────────────────────────
# Pads sequences within each batch to the same length.
# label_pad_token_id=-100 tells the loss function to ignore padding tokens.

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8 if FP16 else None,
)
print("Data collator ready.")

Data collator ready.


In [19]:
# ── Training arguments ────────────────────────────────────────────────────────

training_args = Seq2SeqTrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LEARNING_RATE,
    warmup_steps                = WARMUP_STEPS,
    weight_decay                = WEIGHT_DECAY,
    fp16                        = FP16,
    eval_strategy         = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,
    predict_with_generate       = True,
    generation_max_length       = MAX_TARGET_LEN,
    generation_num_beams        = NUM_BEAMS,
    logging_steps               = 50,
    save_total_limit            = 2,       # keep only 2 checkpoints to save disk
    report_to                   = "none",  # set to "wandb" if you use W&B
    dataloader_num_workers      = 2,
    ddp_find_unused_parameters  = False,
)

print("Training arguments configured.")
eff_batch = BATCH_SIZE * GRAD_ACCUM * max(1, n_gpus)
print(f"Effective batch size: {eff_batch}")
steps_per_epoch = len(train_dataset) // eff_batch
print(f"Steps per epoch    : ~{steps_per_epoch}")
print(f"Total steps        : ~{steps_per_epoch * NUM_EPOCHS}")

Training arguments configured.
Effective batch size: 64
Steps per epoch    : ~222
Total steps        : ~2220


In [23]:
# ── Trainer setup and training ────────────────────────────────────────────────

trainer = Seq2SeqTrainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
   processing_class       = tokenizer,
    data_collator   = data_collator,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Starting adapter training...")
print(f"Training on {len(train_dataset)} samples for up to {NUM_EPOCHS} epochs")
print(f"(Early stopping patience: 2 epochs)")
print("-" * 60)

train_result = trainer.train()

print("-" * 60)
print("Training complete.")
print(f"  Total steps    : {train_result.global_step}")
print(f"  Training loss  : {train_result.training_loss:.4f}")
print(f"  Runtime        : {train_result.metrics.get('train_runtime', 0):.0f}s")

Starting adapter training...
Training on 14252 samples for up to 5 epochs
(Early stopping patience: 2 epochs)
------------------------------------------------------------


Epoch,Training Loss,Validation Loss
1,9.161250,2.738281
2,8.421289,2.353516
3,7.895742,2.222656
4,7.811484,2.189453
5,7.538203,2.140625


------------------------------------------------------------
Training complete.
  Total steps    : 1115
  Training loss  : 8.3451
  Runtime        : 1399s


In [24]:
# ── Save adapter weights ──────────────────────────────────────────────────────
#
# This saves ONLY the LoRA adapter weights (~few MB), not the full model.
# To use later: load base model, then load_adapter() to attach these weights.

adapter_save_path = Path(OUTPUT_DIR) / "final_adapter"
model.save_pretrained(str(adapter_save_path))
tokenizer.save_pretrained(str(adapter_save_path))

# Check sizes
files = list(adapter_save_path.iterdir())
total_mb = sum(f.stat().st_size for f in files if f.is_file()) / 1e6
print(f"Adapter saved to: {adapter_save_path}")
print(f"Total size: {total_mb:.1f} MB")
for f in sorted(files):
    print(f"  {f.name:40s}  {f.stat().st_size/1e6:.2f} MB")

Adapter saved to: /kaggle/working/chittagong_adapter/final_adapter
Total size: 41.4 MB
  README.md                                 0.01 MB
  adapter_config.json                       0.00 MB
  adapter_model.safetensors                 9.46 MB
  tokenizer.json                            31.92 MB
  tokenizer_config.json                     0.00 MB


In [31]:
# ── Generate translations on validation set ───────────────────────────────────
#
# We run generation in batches to avoid OOM on T4.
# The model is set to eval mode and we use beam search.

model.eval()

# Get the forced_bos_token_id for English output
eng_token_id = tokenizer.convert_tokens_to_ids(TGT_LANG)

val_sources  = df_val[DIALECT_COL_VAL].tolist()
val_refs     = df_val[ENGLISH_COL_VAL].tolist()

INFERENCE_BATCH = 16   # can increase if GPU memory allows
predictions = []

print(f"Generating translations for {len(val_sources)} validation samples...")
print(f"(Batch size: {INFERENCE_BATCH}, Beams: {NUM_BEAMS})")

tokenizer.src_lang = SRC_LANG

for i in range(0, len(val_sources), INFERENCE_BATCH):
    batch_texts = val_sources[i : i + INFERENCE_BATCH]

    inputs = tokenizer(
        batch_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_SOURCE_LEN,
    ).to(device)

    with torch.no_grad():
        generated = model.generate(
            **inputs,
            forced_bos_token_id=eng_token_id,
            max_length=MAX_TARGET_LEN,
            num_beams=NUM_BEAMS,
            early_stopping=True,
        )

    decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
    predictions.extend(decoded)

    if (i // INFERENCE_BATCH + 1) % 10 == 0:
        print(f"  {i + INFERENCE_BATCH}/{len(val_sources)} done...")

print(f"\nGeneration complete. {len(predictions)} predictions made.")

# Sanity check: print 5 examples
print("\n--- Sample Predictions ---")
for i in range(min(5, len(val_sources))):
    print(f"\n  [{i+1}]")
    print(f"  Source    : {val_sources[i]}")
    print(f"  Reference : {val_refs[i]}")
    print(f"  Predicted : {predictions[i]}")

Generating translations for 250 validation samples...
(Batch size: 16, Beams: 5)
  160/250 done...

Generation complete. 250 predictions made.

--- Sample Predictions ---

  [1]
  Source    : বাংলাদেশত ৬৪ ইয়ান জেলা
  Reference : 64 districts in Bangladesh
  Predicted : There are 64 districts in Bangladesh.

  [2]
  Source    : আরা বেয়াক্কুন গতহালিয়া বাইরে গেইলাম
  Reference : We all went out yesterday
  Predicted : We both went out yesterday.

  [3]
  Source    : তোইয়ার হতা বলার ধরণ বহুত সুন্দর
  Reference : Your way of speaking is very nice
  Predicted : Your way of saying is very beautiful.

  [4]
  Source    : বরিশালর মানুষ হইল্লে অয় দে?
  Reference : How are the people of Barisal?
  Predicted : Where are the people of Borisha?

  [5]
  Source    : খুলনা জেলা কি বহুত সুন্দর নাকি?
  Reference : Khulna district is very beautiful?
  Predicted : Is Khulna district very beautiful?


In [32]:
# ── Evaluation metrics ────────────────────────────────────────────────────────
#
# We compute all metrics against the validation references.
# References and predictions are cleaned (stripped) before scoring.

refs_clean  = [r.strip() for r in val_refs]
preds_clean = [p.strip() for p in predictions]

# ── 1. SacreBLEU (corpus-level) ───────────────────────────────────────────────
# sacrebleu expects references as a list of lists (one list per reference set)
bleu_result = sacrebleu.corpus_bleu(preds_clean, [refs_clean])
bleu_score  = bleu_result.score

# ── 2. chrF++ ────────────────────────────────────────────────────────────────
# beta=2 gives more weight to recall; word_order=2 gives chrF++
chrf_result = sacrebleu.corpus_chrf(preds_clean, [refs_clean], word_order=2)
chrf_score  = chrf_result.score

# ── 3. ROUGE ─────────────────────────────────────────────────────────────────
scorer_rouge = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"], use_stemmer=False
)
rouge1_scores, rouge2_scores, rougeL_scores = [], [], []
for pred, ref in zip(preds_clean, refs_clean):
    scores = scorer_rouge.score(ref, pred)
    rouge1_scores.append(scores["rouge1"].fmeasure)
    rouge2_scores.append(scores["rouge2"].fmeasure)
    rougeL_scores.append(scores["rougeL"].fmeasure)

rouge1 = np.mean(rouge1_scores) * 100
rouge2 = np.mean(rouge2_scores) * 100
rougeL = np.mean(rougeL_scores) * 100

# ── 4. METEOR ────────────────────────────────────────────────────────────────
meteor_scores = []
for pred, ref in zip(preds_clean, refs_clean):
    # meteor_score expects tokenized lists
    pred_tokens = pred.split()
    ref_tokens  = ref.split()
    if pred_tokens and ref_tokens:
        meteor_scores.append(meteor_score([ref_tokens], pred_tokens))
    else:
        meteor_scores.append(0.0)
meteor = np.mean(meteor_scores) * 100

# ── 5. Word Error Rate (WER) ──────────────────────────────────────────────────
# jiwer computes WER over the entire list at once
wer_score = wer(refs_clean, preds_clean) * 100

# ── 6. Character Error Rate (CER) ────────────────────────────────────────────
cer_score = cer(refs_clean, preds_clean) * 100

print("All metrics computed.")

All metrics computed.


In [33]:
# ── Print evaluation report ───────────────────────────────────────────────────

print("\n" + "=" * 60)
print("  EVALUATION REPORT — CHITTAGONG ADAPTER")
print("=" * 60)
print(f"  Validation samples : {len(val_sources)}")
print(f"  Source lang tag    : {SRC_LANG}")
print(f"  Target lang        : English ({TGT_LANG})")
print(f"  Beam size          : {NUM_BEAMS}")
print("=" * 60)
print()
print("  TRANSLATION QUALITY (higher is better)")
print(f"  {'SacreBLEU (corpus)':30s}: {bleu_score:7.2f}")
print(f"  {'chrF++':30s}: {chrf_score:7.2f}")
print(f"  {'ROUGE-1':30s}: {rouge1:7.2f}")
print(f"  {'ROUGE-2':30s}: {rouge2:7.2f}")
print(f"  {'ROUGE-L':30s}: {rougeL:7.2f}")
print(f"  {'METEOR':30s}: {meteor:7.2f}")
print()
print("  ERROR RATES (lower is better)")
print(f"  {'Word Error Rate (WER)':30s}: {wer_score:7.2f}%")
print(f"  {'Character Error Rate (CER)':30s}: {cer_score:7.2f}%")
print()
print("=" * 60)

# Context: your baseline Chittagong BLEU from full fine-tuning was 11.05
print(f"\n  Baseline BLEU (full fine-tune, no dialect tags): 11.05")
delta = bleu_score - 11.05
direction = "+" if delta >= 0 else ""
print(f"  BLEU delta vs baseline                         : {direction}{delta:.2f}")
print("=" * 60)


  EVALUATION REPORT — CHITTAGONG ADAPTER
  Validation samples : 250
  Source lang tag    : bng_chittagong
  Target lang        : English (eng_Latn)
  Beam size          : 5

  TRANSLATION QUALITY (higher is better)
  SacreBLEU (corpus)            :   18.46
  chrF++                        :   41.41
  ROUGE-1                       :   51.88
  ROUGE-2                       :   32.29
  ROUGE-L                       :   49.96
  METEOR                        :   39.36

  ERROR RATES (lower is better)
  Word Error Rate (WER)         :   75.26%
  Character Error Rate (CER)    :   59.23%


  Baseline BLEU (full fine-tune, no dialect tags): 11.05
  BLEU delta vs baseline                         : +7.41


In [ ]:
# ── Save metrics to JSON ──────────────────────────────────────────────────────

metrics_dict = {
    "dialect"       : "Chittagong",
    "src_lang_tag"  : SRC_LANG,
    "val_samples"   : len(val_sources),
    "num_beams"     : NUM_BEAMS,
    "lora_r"        : LORA_R,
    "lora_alpha"    : LORA_ALPHA,
    "sacrebleu"     : round(bleu_score, 4),
    "chrfpp"        : round(chrf_score, 4),
    "rouge1"        : round(rouge1, 4),
    "rouge2"        : round(rouge2, 4),
    "rougeL"        : round(rougeL, 4),
    "meteor"        : round(meteor, 4),
    "wer"           : round(wer_score, 4),
    "cer"           : round(cer_score, 4),
    "baseline_bleu" : 11.05,
    "bleu_delta"    : round(bleu_score - 11.05, 4),
}

metrics_path = Path(OUTPUT_DIR) / "eval_metrics.json"
metrics_path.parent.mkdir(parents=True, exist_ok=True)
with open(metrics_path, "w") as f:
    json.dump(metrics_dict, f, indent=2)

print(f"Metrics saved to: {metrics_path}")
print(json.dumps(metrics_dict, indent=2))

In [ ]:
# ── Save full predictions to CSV for manual inspection ────────────────────────

df_results = pd.DataFrame({
    "source"    : val_sources,
    "reference" : refs_clean,
    "prediction": preds_clean,
})

# Add per-sentence BLEU and ROUGE-L for detailed analysis
def sentence_bleu(pred, ref):
    try:
        return sacrebleu.sentence_bleu(pred, [ref]).score
    except:
        return 0.0

df_results["sent_bleu"]  = [
    sentence_bleu(p, r) for p, r in zip(preds_clean, refs_clean)
]
df_results["rougeL"]     = rougeL_scores
df_results["meteor"]     = meteor_scores

results_path = Path(OUTPUT_DIR) / "val_predictions.csv"
df_results.to_csv(results_path, index=False, encoding="utf-8-sig")

print(f"Predictions saved to: {results_path}")
print(f"\nTop 5 best translations (by sentence BLEU):")
print(df_results.nlargest(5, "sent_bleu")[["source","reference","prediction","sent_bleu"]].to_string(index=False))
print(f"\nTop 5 worst translations (by sentence BLEU):")
print(df_results.nsmallest(5, "sent_bleu")[["source","reference","prediction","sent_bleu"]].to_string(index=False))

In [ ]:
# ── How to reload the adapter later in a new notebook ────────────────────────

print("""
=============================================================
HOW TO RELOAD THIS ADAPTER IN A NEW NOTEBOOK
=============================================================

1. First, save /kaggle/working/chittagong_adapter as a Kaggle dataset
   (same process as you used for nllb-dialect-model):

   !kaggle datasets create -p /kaggle/working/chittagong_adapter --dir-mode zip

2. In the new notebook, add both datasets as inputs, then load:

   from transformers import NllbTokenizer, AutoModelForSeq2SeqLM
   from peft import PeftModel

   BASE_MODEL  = "/kaggle/input/nllb-dialect-model"
   ADAPTER_DIR = "/kaggle/input/chittagong-adapter/final_adapter"

   tokenizer = NllbTokenizer.from_pretrained(ADAPTER_DIR)
   base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
   model      = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
   model.eval()

3. Run inference:

   tokenizer.src_lang = "bng_chittagong"
   inputs = tokenizer("আঁই হন্ডে যাইচ", return_tensors="pt")
   eng_id = tokenizer.convert_tokens_to_ids("eng_Latn")
   out    = model.generate(**inputs, forced_bos_token_id=eng_id, num_beams=5)
   print(tokenizer.decode(out[0], skip_special_tokens=True))
=============================================================
""")